# Meshroom Reconstruction

To use meshroom dense reconstruction, I downloaded the binaries from the [website](https://github.com/alicevision/meshroom/releases) and ran the reconstruction

I oppened the executable, selected the Photogrametry option, selected the output folder, dropped the images and clicked in run.

The good results are in the textured mesh.

## Visualizing the result

In [1]:
# Convert Meshroom EXR textures to PNG for Open3D
import os
import shutil
from pathlib import Path
import numpy as np

try:
    import imageio.v3 as iio
    print('Using imageio for EXR conversion')
except ImportError:
    raise ImportError('Please install imageio: pip install imageio[all]')

# Path to your Meshroom OBJ file (update as needed)
obj_path = Path('../reconstructions/Meshroom_Segmented/MeshroomCache/Texturing/f397d1bcca84a9348d1c5ff6f4e3656e73e6ceab/texturedMesh.obj')

# Validate core files exist
if not obj_path.exists():
    raise FileNotFoundError(f'OBJ file not found: {obj_path}')

mtl_path = obj_path.with_suffix('.mtl')
if not mtl_path.exists():
    raise FileNotFoundError(f'MTL file not found: {mtl_path}')

def _convert_exr_to_png(exr_path: Path) -> Path:
    """Load EXR texture, tone-map to 8-bit, and write PNG."""
    print(f'Converting {exr_path.name} to PNG...')
    texture = iio.imread(exr_path)
    
    # Handle different data types and convert to 8-bit
    if texture.dtype in (np.float32, np.float64):
        # Tone-map HDR to LDR
        texture = np.clip(texture, 0.0, 1.0)
        texture = (texture * 255.0).astype(np.uint8)
    elif texture.dtype != np.uint8:
        texture = texture.astype(np.uint8)
    
    # Ensure RGB format
    if texture.ndim == 2:
        texture = np.stack([texture]*3, axis=-1)
    elif texture.shape[-1] == 4:
        texture = texture[..., :3]  # Remove alpha channel
    
    png_path = exr_path.with_suffix('.png')
    iio.imwrite(png_path, texture)
    print(f'  → Saved {png_path.name}')
    return png_path

# Update MTL so Open3D can locate PNG textures
with open(mtl_path, 'r', encoding='utf-8') as handle:
    mtl_lines = handle.readlines()

updated = False
for idx, line in enumerate(mtl_lines):
    tokens = line.strip().split(maxsplit=1)
    if tokens and tokens[0].lower() == 'map_kd' and len(tokens) == 2:
        tex_rel_path = tokens[1]
        tex_path = (mtl_path.parent / tex_rel_path).resolve()
        if tex_path.suffix.lower() == '.exr' and tex_path.exists():
            png_path = _convert_exr_to_png(Path(tex_path))
            rel_png = os.path.relpath(png_path, mtl_path.parent).replace('\\', '/')
            mtl_lines[idx] = f'map_Kd {rel_png}\n'
            updated = True

if updated:
    backup_path = mtl_path.with_suffix('.mtl.bak')
    if not backup_path.exists():
        shutil.copy2(mtl_path, backup_path)
    with open(mtl_path, 'w', encoding='utf-8') as handle:
        handle.writelines(mtl_lines)
    print(f'\nConverted EXR textures to PNG and updated MTL.')
    print(f'Original MTL backed up to: {backup_path}')
else:
    print('No EXR textures found to convert.')

Using imageio for EXR conversion
Converting texture_1001.exr to PNG...
  → Saved texture_1001.png

Converted EXR textures to PNG and updated MTL.
Original MTL backed up to: ..\reconstructions\Meshroom_Segmented\MeshroomCache\Texturing\f397d1bcca84a9348d1c5ff6f4e3656e73e6ceab\texturedMesh.mtl.bak
  → Saved texture_1001.png

Converted EXR textures to PNG and updated MTL.
Original MTL backed up to: ..\reconstructions\Meshroom_Segmented\MeshroomCache\Texturing\f397d1bcca84a9348d1c5ff6f4e3656e73e6ceab\texturedMesh.mtl.bak


In [2]:
# Visualize colored OBJ mesh using Open3D
import open3d as o3d
import os

# Path to your Meshroom OBJ file (update as needed)
obj_path = '../reconstructions/Meshroom_Segmented/MeshroomCache/Texturing/f397d1bcca84a9348d1c5ff6f4e3656e73e6ceab/texturedMesh.obj'  # <-- Change if your path is different

# Check if file exists
if not os.path.exists(obj_path):
    raise FileNotFoundError(f'OBJ file not found: {obj_path}')

# Load the mesh (Open3D will try to load textures if referenced in the .mtl file)
mesh = o3d.io.read_triangle_mesh(obj_path, enable_post_processing=True)

# Check if mesh has vertex colors or textures
if mesh.has_vertex_colors():
    print('Mesh has vertex colors.')
elif mesh.has_textures():
    print('Mesh has textures.')
else:
    print('Mesh has no vertex colors or textures.')

# Visualize the mesh
o3d.visualization.draw_geometries([mesh], window_name='Meshroom OBJ Visualization')

Mesh has textures.


In [ ]:
import os
from pathlib import Path
import numpy as np
import imageio.v3 as iio
import open3d as o3d
import gc

obj_path = Path('../reconstructions/Meshroom_Segmented/MeshroomCache/Texturing/f397d1bcca84a9348d1c5ff6f4e3656e73e6ceab/texturedMesh.obj')  # update if needed
out_ply = Path('../reconstructions/meshroom_20_True.ply')

if not obj_path.exists():
    raise FileNotFoundError(f'OBJ file not found: {obj_path}')

# load mesh (Open3D will read .mtl but may not load EXR textures)
mesh = o3d.io.read_triangle_mesh(str(obj_path), enable_post_processing=True)
print(f"Loaded mesh: {len(mesh.vertices)} vertices, {len(mesh.triangles)} faces")

# try to find texture referenced in .mtl
mtl_path = obj_path.with_suffix('.mtl')
texture_files = []
if mtl_path.exists():
    for line in mtl_path.read_text(encoding='utf-8', errors='ignore').splitlines():
        tokens = line.strip().split(maxsplit=1)
        if tokens and tokens[0].lower() == 'map_kd' and len(tokens) == 2:
            tex = (mtl_path.parent / tokens[1].strip()).resolve()
            if tex.exists():
                texture_files.append(tex)
            else:
                # try same name with .png if EXR was replaced earlier
                alt = tex.with_suffix('.png')
                if alt.exists():
                    texture_files.append(alt)

if len(texture_files) == 0:
    print("No texture file found in MTL. Attempting to use mesh textures (if any).")

# prefer first texture
tex_path = Path(texture_files[0]) if texture_files else None
if tex_path:
    print(f"Using texture: {tex_path}")
else:
    print("No usable texture found. Will attempt point sampling fallback.")

# helper: bilinear sample from image at uv (u,v in [0,1])
def sample_texture_bilinear(img, uv):
    h, w = img.shape[:2]
    u = np.clip(uv[:, 0], 0.0, 1.0) * (w - 1)
    v = np.clip(uv[:, 1], 0.0, 1.0) * (h - 1)
    # v origin correction (OBJ UV convention -> v=0 bottom); most images top-down so invert v
    v = (h - 1) - v
    x0 = np.floor(u).astype(np.int32)
    x1 = np.clip(x0 + 1, 0, w - 1)
    y0 = np.floor(v).astype(np.int32)
    y1 = np.clip(y0 + 1, 0, h - 1)
    wx = u - x0
    wy = v - y0
    c00 = img[y0, x0]
    c10 = img[y0, x1]
    c01 = img[y1, x0]
    c11 = img[y1, x1]
    c0 = c00 * (1 - wx)[:, None] + c10 * wx[:, None]
    c1 = c01 * (1 - wx)[:, None] + c11 * wx[:, None]
    c = c0 * (1 - wy)[:, None] + c1 * wy[:, None]
    return c.astype(np.float32) / 255.0

def bake_vertex_colors_from_texture(mesh, tex_img_path, batch_faces=50000):
    # ensure UVs exist
    if not hasattr(mesh, "triangle_uvs") or len(mesh.triangle_uvs) == 0:
        raise RuntimeError("Mesh has no triangle UVs (triangle_uvs). Cannot bake.")
    # load texture
    img = iio.imread(str(tex_img_path))
    if img.dtype != np.uint8:
        img = np.clip(img, 0.0, 1.0)
        img = (img * 255).astype(np.uint8)
    if img.ndim == 2:
        img = np.stack([img]*3, axis=-1)
    if img.shape[2] == 4:
        img = img[..., :3]
    img_h, img_w = img.shape[:2]
    # arrays
    triangles = np.asarray(mesh.triangles, dtype=np.int32)
    tri_uvs = np.asarray(mesh.triangle_uvs, dtype=np.float32).reshape((-1, 3, 2))  # (n_tri,3,2)
    n_vertices = len(mesh.vertices)
    sums = np.zeros((n_vertices, 3), dtype=np.float64)
    counts = np.zeros((n_vertices,), dtype=np.int32)

    n_tri = triangles.shape[0]
    for start in range(0, n_tri, batch_faces):
        end = min(n_tri, start + batch_faces)
        tris = triangles[start:end]
        uvs = tri_uvs[start:end].reshape((-1, 2))  # flattened per-vertex
        # sample colors for each triangle vertex
        colors = sample_texture_bilinear(img, uvs)  # shape (batch_faces*3, 3)
        colors = colors.reshape((-1, 3))  # (n_batch*3,3)
        # accumulate per vertex
        for i in range(end - start):
            tri = tris[i]
            c0 = colors[3*i + 0]
            c1 = colors[3*i + 1]
            c2 = colors[3*i + 2]
            sums[tri[0]] += c0
            sums[tri[1]] += c1
            sums[tri[2]] += c2
            counts[tri[0]] += 1
            counts[tri[1]] += 1
            counts[tri[2]] += 1
        gc.collect()
        print(f"Processed triangles {start}..{end} / {n_tri}")

    # avoid division by zero
    mask = counts > 0
    vertex_colors = np.zeros((n_vertices, 3), dtype=np.float32)
    vertex_colors[mask] = (sums[mask] / counts[mask][:, None]).astype(np.float32)
    # for vertices with zero counts, set to gray
    vertex_colors[~mask] = 0.5
    return vertex_colors

# Try baking if possible
try:
    if tex_path and len(mesh.triangle_uvs) > 0:
        vcols = bake_vertex_colors_from_texture(mesh, tex_path)
        mesh.vertex_colors = o3d.utility.Vector3dVector(vcols)
        o3d.io.write_triangle_mesh(str(out_ply), mesh, write_vertex_colors=True)
        print(f"Saved colored mesh to: {out_ply}")
    else:
        raise RuntimeError("No texture or UVs available for baking.")
except Exception as e:
    print(f"Baking failed: {e}")
    print("Falling back to sampling colored point cloud from textured mesh (slower but robust).")
    try:
        # fallback: sample points with colors (requires texture accessible by Open3D)
        pcd = mesh.sample_points_poisson_disk(number_of_points=300000)
        if pcd.has_colors():
            o3d.io.write_point_cloud(str(out_ply), pcd, write_ascii=False)
            print(f"Saved colored point cloud to: {out_ply}")
        else:
            # final fallback: save geometry only
            o3d.io.write_triangle_mesh(str(out_ply), mesh, write_vertex_colors=False)
            print(f"Saved mesh without colors to: {out_ply} (no colors available)")
    except Exception as e2:
        print(f"Fallback also failed: {e2}")
        # still attempt to write mesh
        o3d.io.write_triangle_mesh(str(out_ply), mesh, write_vertex_colors=False)
        print(f"Wrote mesh without colors to: {out_ply}")

# Visualize the resulting PLY (or mesh)
if out_ply.exists():
    geom = o3d.io.read_triangle_mesh(str(out_ply))
    print("Result mesh info:", len(geom.vertices), "vertices,", len(geom.triangles), "faces")
    if geom.has_vertex_colors():
        print("Result has vertex colors.")
    elif geom.has_textures():
        print("Result has textures.")
    else:
        print("Result has no vertex colors or textures.")
    o3d.visualization.draw_geometries([geom], window_name='TexturedMesh (baked)')

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loaded mesh: 1068321 vertices, 356135 faces
Using texture: C:\Users\gnoceras\Documents\GustavoPersonal\ReconstructionStudies\reconstructions\meshroom_20_True\texture_1001.png
Loaded mesh: 1068321 vertices, 356135 faces
Using texture: C:\Users\gnoceras\Documents\GustavoPersonal\ReconstructionStudies\reconstructions\meshroom_20_True\texture_1001.png
Processed triangles 0..50000 / 356135
Processed triangles 0..50000 / 356135
Processed triangles 50000..100000 / 356135
Processed triangles 50000..100000 / 356135
Processed triangles 100000..150000 / 356135
Processed triangles 100000..150000 / 356135
Processed triangles 150000..200000 / 356135
Processed triangles 150000..200000 / 356135
Processed triangles 200000..250000 / 356135
Processed triangles 200000..250000 / 356135
Processed triangles 250000..300000 / 356135
Processed t

## Remarks

The reconstruction is absolutely wild: 9/10. The downsides are: heavy software, need of a GPU, lack of easiness to export the colored textured output.